# IT Operations Intelligence
## Fase 5: Transformación de Datos y Feature Engineering

**Objetivo:** Transformar las variables del dataset original mediante normalización, codificación y creación de nuevas características que permitan mejorar el rendimiento de los modelos predictivos.

In [ ]:
import pandas as pd # Librería para manipulación y análisis de datos en DataFrames
import numpy as np # Librería para operaciones numéricas y manejo de arreglos multidimensionales
import os # Módulo para interactuar con el sistema operativo (rutas, directorios)


In [ ]:
# Cargar el dataset desde un archivo CSV ubicado en el directorio raw
df = pd.read_csv("../data/raw/incident_event_log.csv")  # Leer el archivo CSV y almacenarlo en un DataFrame


In [ ]:
# Mostrar las primeras filas del dataset para inspeccionar visualmente su estructura y contenido
df.head()  # Visualizar las primeras 5 filas del DataFrame


,number,incident_state,active,reassignment_count,reopen_count,sys_mod_count,made_sla,caller_id,opened_by,opened_at,...,u_priority_confirmation,notify,problem_id,rfc,vendor,caused_by,closed_code,resolved_by,resolved_at,closed_at
0,INC0000045,New,True,0,0,0,True,Caller 2403,Opened by 8,29/2/2016 01:16,...,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
1,INC0000045,Resolved,True,0,0,2,True,Caller 2403,Opened by 8,29/2/2016 01:16,...,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
2,INC0000045,Resolved,True,0,0,3,True,Caller 2403,Opened by 8,29/2/2016 01:16,...,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
3,INC0000045,Closed,False,0,0,4,True,Caller 2403,Opened by 8,29/2/2016 01:16,...,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
4,INC0000047,New,True,0,0,0,True,Caller 2403,Opened by 397,29/2/2016 04:40,...,False,Do Not Notify,?,?,?,?,code 5,Resolved by 81,1/3/2016 09:52,6/3/2016 10:00


##### Análisis de las primeras filas del dataset
Se muestran las primeras 5 filas del conjunto de datos con sus 36 columnas originales. Podemos observar variables como el identificador del incidente (number), el estado (incident_state), fechas de apertura y resolución, así como indicadores de SLA, prioridad, impacto y urgencia. La presencia de valores '?' en columnas como problem_id, rfc y vendor indica datos faltantes que requerirán tratamiento posterior. También se aprecia que un mismo incidente (INC0000045) puede tener múltiples registros correspondientes a diferentes estados (New, Resolved, Closed).

### Transformación de Variables Temporales

In [ ]:
# Definir las columnas de fecha a convertir
date_cols = ["opened_at", "resolved_at", "closed_at", "sys_created_at", "sys_updated_at"]

# Iterar sobre cada columna de fecha
for col in date_cols:

    # Verificar que la columna exista en el DataFrame
    if col in df.columns:

        # Convertir la columna a tipo datetime con el formato especificado
        df[col] = pd.to_datetime(df[col], format="%d/%m/%Y %H:%M", errors="coerce")

        # Mostrar el tipo de dato resultante para confirmar la conversión
        print(f"{col}: {df[col].dtype}")


opened_at: datetime64[us]
resolved_at: datetime64[us]
closed_at: datetime64[us]
sys_created_at: datetime64[us]
sys_updated_at: datetime64[us]


##### Análisis de la conversión de fechas
Todas las columnas temporales fueron convertidas exitosamente al tipo datetime64[us]. Esto permite realizar operaciones avanzadas como extracción de componentes (hora, día, mes, año) y cálculos de diferencias temporales entre fechas, como el tiempo de resolución de incidentes. El formato origen 'dd/mm/aaaa HH:MM' fue interpretado correctamente.

In [5]:
# Verificar que la columna opened_at exista en el DataFrame
if "opened_at" in df.columns:

    # Extraer la hora del día (0-23) en que se abrió el incidente
    df["open_hour"] = df["opened_at"].dt.hour

    # Extraer el día de la semana (0=lunes, 6=domingo)
    df["open_dayofweek"] = df["opened_at"].dt.dayofweek

    # Extraer el mes del año (1-12) en que se abrió el incidente
    df["open_month"] = df["opened_at"].dt.month

    # Extraer el año en que se abrió el incidente
    df["open_year"] = df["opened_at"].dt.year

    # Mensaje de confirmación
    print("Temporal features created from opened_at")

    # Mostrar las primeras filas de las nuevas columnas
    print(df[["open_hour", "open_dayofweek", "open_month", "open_year"]].head())


Temporal features created from opened_at
   open_hour  open_dayofweek  open_month  open_year
0          1               0           2       2016
1          1               0           2       2016
2          1               0           2       2016
3          1               0           2       2016
4          4               0           2       2016


##### Análisis de características temporales extraídas
Se crearon 4 nuevas variables a partir de la fecha de apertura: hora del día, día de la semana, mes y año. Estos atributos permitirán identificar patrones temporales en la apertura de incidentes, como horas pico, días con mayor carga de trabajo o tendencias estacionales a lo largo del año. Por ejemplo, los primeros registros muestran aperturas a la 1:00 y 4:00 AM en el mes de febrero de 2016, todos en día lunes (0).

In [6]:
# Verificar que ambas columnas necesarias existan
if "opened_at" in df.columns and "resolved_at" in df.columns:

    # Calcular la diferencia en horas entre la resolución y la apertura
    df["resolution_time_hours"] = (
        df["resolved_at"] - df["opened_at"]
    ).dt.total_seconds() / 3600

    # Reemplazar valores negativos con 0 para mantener coherencia
    df["resolution_time_hours"] = df["resolution_time_hours"].clip(0)

    # Mensaje de confirmación
    print("Tiempo de resolución calculado")

    # Mostrar estadísticas descriptivas del tiempo de resolución
    print(df["resolution_time_hours"].describe())


Tiempo de resolución calculado
count    138571.000000
mean        269.596262
std         650.867377
min           0.000000
25%           4.066667
50%          73.516667
75%         262.183333
max        8070.166667
Name: resolution_time_hours, dtype: float64


##### Análisis del tiempo de resolución
El tiempo promedio de resolución es de aproximadamente 270 horas (~11 días), con una alta desviación estándar de 651 horas, indicando una gran variabilidad entre incidentes. La mediana es de 73.5 horas (~3 días), lo que significa que el 50% de los incidentes se resuelven en menos de 3 días. El percentil 75 es de 262 horas (~11 días), y el valor máximo alcanza 8070 horas (~336 días), lo que sugiere la presencia de incidentes extremadamente prolongados que podrían considerarse valores atípicos. Se cuenta con 138,571 observaciones válidas para esta métrica.

### Codificación de Variables Categóricas

In [7]:
# Mapeo ordinal de prioridad: a mayor número, mayor prioridad
priority_map = {"1 - Critical": 4, "2 - High": 3, "3 - Moderate": 2, "4 - Low": 1}

# Mapeo ordinal de impacto: 3=Alto, 2=Medio, 1=Bajo
impact_map = {"1 - High": 3, "2 - Medium": 2, "3 - Low": 1}

# Mapeo ordinal de urgencia: 3=Alta, 2=Media, 1=Baja
urgency_map = {"1 - High": 3, "2 - Medium": 2, "3 - Low": 1}

# Verificar que la columna priority exista y aplicar el mapeo ordinal
if "priority" in df.columns:
    df["priority_score"] = df["priority"].map(priority_map)

# Verificar que la columna impact exista y aplicar el mapeo ordinal
if "impact" in df.columns:
    df["impact_score"] = df["impact"].map(impact_map)

# Verificar que la columna urgency exista y aplicar el mapeo ordinal
if "urgency" in df.columns:
    df["urgency_score"] = df["urgency"].map(urgency_map)

# Encabezado del reporte de distribución
print("Variables ordinales codificadas:")

# Iterar sobre las nuevas columnas numéricas
for col in ["priority_score", "impact_score", "urgency_score"]:

    # Verificar que la columna exista
    if col in df.columns:

        # Mostrar la frecuencia de cada valor codificado
        print(f"{col}: {df[col].value_counts().to_dict()}")


Variables ordinales codificadas:
priority_score: {2: 132452, 1: 4030, 3: 2972, 4: 2258}
impact_score: {2: 134335, 1: 3886, 3: 3491}
urgency_score: {2: 134094, 3: 4020, 1: 3598}


##### Análisis de la codificación ordinal
Las variables ordinales fueron transformadas exitosamente a valores numéricos respetando su orden intrínseco. En priority_score, el valor dominante es 2 (Moderate) con 132,452 registros, seguido de 1 (Low) con 4,030. En impacto y urgencia, el valor 2 (Medium) también predomina con ~134,000 registros cada uno. Esto indica que la mayoría de los incidentes tienen criticidad moderada, con una minoría en los extremos (alta o baja). Esta codificación preserva la relación ordinal y permitirá que los modelos de machine learning aprovechen esta información.

### Creación de Indicadores Compuestos

In [8]:
# Verificar que las 3 columnas de puntuación existan
if all(c in df.columns for c in ["impact_score", "urgency_score", "priority_score"]):

    # Calcular el promedio de las tres puntuaciones (rango teórico 1-4)
    df["criticality_score"] = (
        df["impact_score"] + df["urgency_score"] + df["priority_score"]
    ) / 3

    # Mensaje de confirmación
    print("Criticality Score creado")

    # Mostrar estadísticas descriptivas del score compuesto
    print(df["criticality_score"].describe())


Criticality Score creado
count    141712.000000
mean          2.008197
std           0.252330
min           1.000000
25%           2.000000
50%           2.000000
75%           2.000000
max           3.333333
Name: criticality_score, dtype: float64


##### Análisis del Criticality Score compuesto
El score de criticidad compuesto tiene un promedio de 2.01 con una desviación estándar muy baja de 0.25, lo que indica poca variabilidad. La mediana y los percentiles 25% y 75% son todos 2.0, reflejando una concentración masiva de incidentes con criticidad media. El valor mínimo es 1.0 y el máximo 3.33, correspondiente a incidentes que combinan los niveles más altos en las tres dimensiones. Esta métrica resume en una sola variable la criticidad global del incidente.

In [9]:
# Verificar que la columna del score compuesto exista
if "criticality_score" in df.columns:

    # Discretizar el score continuo en intervalos con etiquetas descriptivas
    df["criticality_level"] = pd.cut(
        df["criticality_score"],
        bins=[0, 1.5, 2.5, 4],
        labels=["Low", "Medium", "High"],
        include_lowest=True
    )

    # Encabezado del reporte de niveles
    print("Nivel de criticidad:")

    # Mostrar la cantidad de incidentes por cada nivel de criticidad
    print(df["criticality_level"].value_counts())


Nivel de criticidad:
criticality_level
Medium    132452
High        5230
Low         4030
Name: count, dtype: int64


##### Análisis de la clasificación por nivel de criticidad
La clasificación revela una distribución altamente desbalanceada: 132,452 incidentes (93.5%) son de criticidad media (Medium), mientras que solo 5,230 son altos (High) y 4,030 son bajos (Low). Esto refleja que la mayoría de los tickets de soporte tienen un perfil de criticidad estándar. Los modelos predictivos deberán considerar este desbalance para evitar sesgos hacia la clase mayoritaria, posiblemente mediante técnicas de remuestreo o ponderación.

In [10]:
# Crear columna binaria: más de 2 reasignaciones O al menos una reapertura
df["problematic_ticket"] = (
    (df["reassignment_count"] > 2) | (df["reopen_count"] > 0)
).astype(int)

# Encabezado del reporte de tickets problemáticos
print("Tickets problemáticos:")

# Mostrar cuántos tickets son problemáticos (1) vs normales (0)
print(df["problematic_ticket"].value_counts())


Tickets problemáticos:
problematic_ticket
0    120720
1     20992
Name: count, dtype: int64


##### Análisis de tickets problemáticos
Se identificaron 20,992 tickets problemáticos (14.8% del total) frente a 120,720 normales. Estos tickets representan incidentes que requirieron múltiples reasignaciones o fueron reabiertos, lo cual es un indicador de problemas complejos o mal gestionados. Esta variable es especialmente valiosa como característica predictiva o como variable objetivo para identificar incidentes que requieren atención especial o procesos de mejora.

In [11]:
# Transformar la variable booleana made_sla a numérica (True→1, False→0)
df["made_sla_num"] = df["made_sla"].astype(int)


### Guardado del Dataset Transformado

In [12]:
# Crear el directorio de salida processed si no existe
os.makedirs("../data/processed", exist_ok=True)

# Exportar el DataFrame transformado a CSV sin incluir el índice
df.to_csv("../data/processed/incident_event_log_transformado.csv", index=False)

# Mensaje de confirmación con la ruta del archivo guardado
print("Dataset transformado guardado como 'incident_event_log_transformado.csv'")


Dataset transformado guardado como 'incident_event_log_transformado.csv'


##### Análisis del guardado del dataset transformado
El dataset transformado ha sido exportado exitosamente al directorio `../data/processed/` con el nombre `incident_event_log_transformado.csv`. Este archivo contiene todas las nuevas características creadas durante el proceso de feature engineering y estará disponible para las fases posteriores de modelado y evaluación de los algoritmos predictivos.

In [13]:
# Cargar el dataset original desde el directorio raw
df_original = pd.read_csv("../data/raw/incident_event_log.csv")

# Cargar el dataset transformado desde el directorio processed
df_transformado = pd.read_csv("../data/processed/incident_event_log_transformado.csv")

# Línea decorativa superior
print("=" * 50)

# Encabezado de la sección de dimensiones
print("DIMENSIONES")

# Línea decorativa inferior
print("=" * 50)

# Mostrar filas y columnas del dataset original
print(f"Original: {df_original.shape}")

# Mostrar filas y columnas del dataset transformado
print(f"Transformado: {df_transformado.shape}")

# Salto de línea entre secciones
print()

# Línea decorativa superior
print("=" * 50)

# Encabezado de la sección de columnas añadidas
print("COLUMNAS AÑADIDAS")

# Línea decorativa inferior
print("=" * 50)

# Calcular las columnas que están solo en el dataset transformado
columnas_nuevas = set(df_transformado.columns) - set(df_original.columns)

# Iterar sobre las columnas nuevas ordenadas alfabéticamente
for col in sorted(columnas_nuevas):

    # Imprimir el nombre de cada columna añadida
    print(col)

# Salto de línea entre secciones
print()

# Verificar que no se eliminaron columnas accidentalmente durante la transformación
col_eliminadas = set(df_original.columns) - set(df_transformado.columns)

# Mostrar la cantidad de columnas eliminadas
print(f"Columnas eliminadas: {len(col_eliminadas)}")

# Mostrar los nombres de columnas eliminadas o indicar que no hubo
print(col_eliminadas if col_eliminadas else "Ninguna")


DIMENSIONES
Original: (141712, 36)
Transformado: (141712, 48)

COLUMNAS AÑADIDAS
criticality_level
criticality_score
impact_score
made_sla_num
open_dayofweek
open_hour
open_month
open_year
priority_score
problematic_ticket
resolution_time_hours
urgency_score

Columnas eliminadas: 0
Ninguna


##### Análisis comparativo entre dataset original y transformado
El dataset original tenía 36 columnas y 141,712 filas, mientras que el transformado cuenta con 48 columnas, manteniendo el mismo número de filas. Se añadieron 12 nuevas variables: 4 temporales (open_hour, open_dayofweek, open_month, open_year), 3 codificaciones ordinales (priority_score, impact_score, urgency_score), 2 indicadores compuestos (criticality_score, criticality_level), 1 métrica operacional (resolution_time_hours), 1 variable binaria (problematic_ticket) y 1 variable objetivo numérica (made_sla_num). No se eliminó ninguna columna original, preservando toda la información inicial. Este conjunto enriquecido de características permitirá entrenar modelos más robustos y precisos.